In [300]:
import tech.tablesaw.api.Table;
import tech.tablesaw.api.ColumnType;
import tech.tablesaw.io.csv.CsvReadOptions;

String path ="C:\\Users\\thkle\\SSE554\\SSE554-Capstone-Project\\data\\1000_items_catalog_v2.csv";

/*Previously ran code to verify that data types do not violate min and maxes of numeric types listed
    Column |        Min |        Max
    price | 9.619999885559082 | 798.0599975585938
    review_score |        1.0 |        5.0
    review_count |       20.0 |     9989.0
    stock_quantity |        0.0 |     1998.0
*/
CsvReadOptions options = CsvReadOptions.builder(path)
    .columnTypesPartial(Map.of(
        "price", ColumnType.FLOAT,
        "review_score", ColumnType.FLOAT,
        "review_count", ColumnType.SHORT,
        "stock_quantity", ColumnType.SHORT
    )).build();
System.out.println( options );

Table table = Table.read().csv(options);
System.out.println(table.structure());


tech.tablesaw.io.csv.CsvReadOptions@a606c301
   Structure of 1000_items_catalog_v2.csv   
 Index  |   Column Name    |  Column Type  |
--------------------------------------------
     0  |       image_url  |       STRING  |
     1  |            name  |       STRING  |
     2  |       publisher  |       STRING  |
     3  |     description  |       STRING  |
     4  |        category  |       STRING  |
     5  |            tags  |       STRING  |
     6  |           price  |        FLOAT  |
     7  |    review_score  |        FLOAT  |
     8  |    review_count  |        SHORT  |
     9  |  stock_quantity  |        SHORT  |
    10  |      date_added  |   LOCAL_DATE  |


In [301]:
System.out.println("Row count: " + table.rowCount());
for( int x = 0; x < table.columnCount(); x++)
    System.out.println("Column " + x + ": " + table.column(x).name() + " - " + table.column(x).unique().size() + " unique values");

Row count: 1000
Column 0: image_url - 1000 unique values
Column 1: name - 1000 unique values
Column 2: publisher - 8 unique values
Column 3: description - 1000 unique values
Column 4: category - 6 unique values
Column 5: tags - 999 unique values
Column 6: price - 994 unique values
Column 7: review_score - 370 unique values
Column 8: review_count - 953 unique values
Column 9: stock_quantity - 781 unique values
Column 10: date_added - 883 unique values


In [302]:
/*
//Find min and max of numeric columns to determine if we can change to smaller types to save memory
import tech.tablesaw.api.ColumnType;
import java.util.ArrayList;
import java.util.Arrays;
import tech.tablesaw.api.NumericColumn;

ArrayList<ColumnType> numericTypes = new ArrayList<>(Arrays.asList(ColumnType.DOUBLE, ColumnType.FLOAT, ColumnType.INTEGER, ColumnType.LONG, ColumnType.SHORT));
System.out.printf("%10s | %10s | %10s | %10s%n", "Column", "Min", "Max", "Type");
for(int x = 0; x < table.columnCount(); x++) {
    if(numericTypes.contains(table.column(x).type())){   //Only go over number columns
        NumericColumn<?> numCol = (NumericColumn<?>) table.column(x);
        System.out.printf("%10s | %10s | %10s | %10s%n", numCol.name(), numCol.min(), numCol.max(), numCol.type());
    }
}
*/

In [303]:
/*
//Results above show that we can alter the doubles to floats and integers to shorts.
import tech.tablesaw.api.FloatColumn;
import tech.tablesaw.api.ShortColumn;
import tech.tablesaw.api.IntColumn;
import tech.tablesaw.api.DoubleColumn;

FloatColumn priceColFloat = table.doubleColumn("price").asFloatColumn();
FloatColumn reviewScoreColFloat = table.doubleColumn("review_score").asFloatColumn();
ShortColumn reviewCountColShort = table.intColumn("review_count").asShortColumn();
ShortColumn stockQuantityColShort = table.intColumn("stock_quantity").asShortColumn();

table.removeColumns("price", "review_score", "review_count", "stock_quantity");
table.addColumns(priceColFloat, reviewScoreColFloat, reviewCountColShort, stockQuantityColShort);
System.out.println(table.structure());

//Get numerics for first row and print types to verify
System.out.println("First row price type: " + table.floatColumn("price").get(0).getClass());
System.out.println("First row review score type: " + table.floatColumn("review_score").get(0).getClass());
System.out.println("First row review count type: " + table.shortColumn("review_count").get(0).getClass());
System.out.println("First row stock quantity type: " + table.shortColumn("stock_quantity").get(0).getClass());
*/

In [304]:
//Add an id column (short is enough for 1000 rows)
import tech.tablesaw.api.ShortColumn;
import java.util.Set;
import java.util.HashSet;

Set<Short> uniqueIds = new HashSet<>();
short min = 0;
short max = Short.MAX_VALUE;
for (int i = 0; i < table.rowCount(); i++) {
    while(true) {
        short id = (short) (Math.random() * (max - min + 1) + min); // Generate random ID between min and max
        if (!uniqueIds.contains(id)) {
            uniqueIds.add(id);
            break;
        }
    }
}
ShortColumn idCol = ShortColumn.create("id", uniqueIds.stream() );
table.addColumns(idCol); // Add the column

System.out.println("Added ID column successfully!");
System.out.println("Table now has " + table.columnCount() + " columns");
System.out.println("ID column sample: " + table.shortColumn("id").get(0) + ", " + 
                   table.shortColumn("id").get(1) + ", " + table.shortColumn("id").get(2));
System.out.println("\nUpdated table structure:");
System.out.println(table.structure());
System.out.println(table.first(5)); // Display the first 5 rows of the table
table.write().csv("C:\\Users\\thkle\\SSE554\\SSE554-Capstone-Project\\data\\1000_items_catalog_v2_optimized.csv");

Added ID column successfully!
Table now has 12 columns
ID column sample: 18437, 24582, 10248

Updated table structure:
   Structure of 1000_items_catalog_v2.csv   
 Index  |   Column Name    |  Column Type  |
--------------------------------------------
     0  |       image_url  |       STRING  |
     1  |            name  |       STRING  |
     2  |       publisher  |       STRING  |
     3  |     description  |       STRING  |
     4  |        category  |       STRING  |
     5  |            tags  |       STRING  |
     6  |           price  |        FLOAT  |
     7  |    review_score  |        FLOAT  |
     8  |    review_count  |        SHORT  |
     9  |  stock_quantity  |        SHORT  |
    10  |      date_added  |   LOCAL_DATE  |
    11  |              id  |        SHORT  |
                                                                                                                                                                                                              

In [305]:
// Get individual unique tags (not tag combinations)
import java.util.Set;
import java.util.HashSet;
import java.util.Map;
import java.util.HashMap;
import java.util.Arrays;
import tech.tablesaw.api.StringColumn;

Set<String> allIndividualTags = new HashSet<>();
StringColumn tagsCol = table.stringColumn("tags");

System.out.println("Processing " + tagsCol.size() + " rows to extract individual tags...\n");

// Split each tag combination and collect individual tags
for (String tags : tagsCol)
    allIndividualTags.addAll(Arrays.asList(tags.split(" ")));

System.out.println("Individual unique tags (" + allIndividualTags.size() + " total):");
allIndividualTags.stream().forEach(tag -> System.out.println("  • " + tag));

Processing 1000 rows to extract individual tags...

Individual unique tags (16 total):
  • ergonomic
  • rechargeable
  • compact
  • portable
  • versatile
  • newarrival
  • lightweight
  • giftable
  • durable
  • premium
  • modern
  • bestseller
  • limitededition
  • wireless
  • waterproof
  • essential


In [306]:
// Get individual unique publishers
import java.util.Set;
import java.util.HashSet;
import java.util.Map;
import java.util.HashMap;
import java.util.Arrays;
import tech.tablesaw.api.StringColumn;

Set<String> allIndividualPublishers = new HashSet<>();
StringColumn publishersCol = table.stringColumn("publisher");
StringColumn uniquePublishersCol = publishersCol.unique();
System.out.println("Processing " + uniquePublishersCol.size() + " unique publishers...");
uniquePublishersCol.asList().stream().forEach(tag -> System.out.println("  • " + tag));


Processing 8 unique publishers...
  • NorthPeak
  • UrbanNest
  • Summit Gear Co.
  • SilverLine Electronics
  • Maple Street Press
  • Horizon Tech
  • BlueRiver Outfitters
  • BrightLeaf Publishing


In [307]:
// Get individual unique categories
StringColumn categoryCol = table.stringColumn("category");
StringColumn uniqueCategoriesCol = categoryCol.unique();

System.out.println("Processing " + uniqueCategoriesCol.size() + " unique categories...");
uniqueCategoriesCol.asList().stream().forEach(category -> System.out.println("  • " + category));

Processing 6 unique categories...
  • Home & Kitchen
  • Clothing
  • Office Supplies
  • Electronics
  • Books
  • Sports & Outdoors


In [308]:
// Count frequency of each publisher, category, and individual tag
import java.util.Map;
import java.util.HashMap;
import java.util.Arrays;
import tech.tablesaw.api.StringColumn;

System.out.println("📊 FREQUENCY ANALYSIS");
System.out.println("====================\n");

// 1. Publisher frequency counts
System.out.println("🏢 PUBLISHER FREQUENCIES:");
StringColumn publisherCol = table.stringColumn("publisher");
Map<String, Integer> publisherCounts = new HashMap<>();

for (String publisher : publisherCol) {
    publisherCounts.put(publisher, publisherCounts.getOrDefault(publisher, 0) + 1);
}

System.out.println("\nFound " + publisherCounts.size() + " unique publishers. Showing all:");
publisherCounts.entrySet().stream()
    .sorted(Map.Entry.<String, Integer>comparingByValue().reversed())
    .forEach(entry -> 
        System.out.println("  " + entry.getKey() + ": " + entry.getValue() + " items")
    );

// 2. Category frequency counts  
System.out.println("\n📦 CATEGORY FREQUENCIES:");
StringColumn categoryCol = table.stringColumn("category");
Map<String, Integer> categoryCounts = new HashMap<>();

for (String category : categoryCol) {
    categoryCounts.put(category, categoryCounts.getOrDefault(category, 0) + 1);
}

System.out.println("\nFound " + categoryCounts.size() + " unique categories. Showing all:");
categoryCounts.entrySet().stream()
    .sorted(Map.Entry.<String, Integer>comparingByValue().reversed())
    .forEach(entry -> 
        System.out.println("  " + entry.getKey() + ": " + entry.getValue() + " items")
    );

// 3. Individual tag frequency counts
System.out.println("\n🏷️  TAG FREQUENCIES (All " + "tags" + "):");
StringColumn tagsCol = table.stringColumn("tags");
Map<String, Integer> tagCounts = new HashMap<>();

for (String tagCombination : tagsCol) {
    if (tagCombination != null && !tagCombination.trim().isEmpty()) {
        String[] individualTags = tagCombination.trim().split("\\s+");
        for (String tag : individualTags) {
            String cleanTag = tag.trim();
            if (!cleanTag.isEmpty()) {
                tagCounts.put(cleanTag, tagCounts.getOrDefault(cleanTag, 0) + 1);
            }
        }
    }
}

System.out.println("Found " + tagCounts.size() + " unique tags. Showing all:");
tagCounts.entrySet().stream()
    .sorted(Map.Entry.<String, Integer>comparingByValue().reversed())
    .forEach(entry -> 
        System.out.println("  " + entry.getKey() + ": " + entry.getValue() + " times")
    );

System.out.println("\n📈 SUMMARY STATISTICS:");
System.out.println("  Total unique publishers: " + publisherCounts.size());
System.out.println("  Total unique categories: " + categoryCounts.size()); 
System.out.println("  Total unique tags: " + tagCounts.size());

📊 FREQUENCY ANALYSIS

🏢 PUBLISHER FREQUENCIES:

Found 8 unique publishers. Showing all:
  NorthPeak: 139 items
  BrightLeaf Publishing: 138 items
  Summit Gear Co.: 129 items
  BlueRiver Outfitters: 126 items
  Horizon Tech: 123 items
  UrbanNest: 121 items
  Maple Street Press: 113 items
  SilverLine Electronics: 111 items

📦 CATEGORY FREQUENCIES:

Found 6 unique categories. Showing all:
  Home & Kitchen: 188 items
  Office Supplies: 171 items
  Sports & Outdoors: 167 items
  Electronics: 163 items
  Books: 161 items
  Clothing: 150 items

🏷️  TAG FREQUENCIES (All tags):
Found 16 unique tags. Showing all:
  limitededition: 346 times
  giftable: 333 times
  premium: 321 times
  modern: 319 times
  bestseller: 315 times
  essential: 314 times
  portable: 312 times
  durable: 311 times
  rechargeable: 310 times
  lightweight: 309 times
  compact: 308 times
  newarrival: 306 times
  ergonomic: 305 times
  versatile: 305 times
  wireless: 302 times
  waterproof: 284 times

📈 SUMMARY STATIS